# 02 : Variables y validación temporal

Este notebook construye variables disponibles al origen y comprueba que los periodos de entrenamiento y validación se mantengan separados. La comparación principal usa `asof_v1`.

## Ejecución de la entrega

Ejecuta todas las celdas en orden en una sesión limpia. El modo predeterminado
`full` utiliza datos reales; el notebook 04 requiere GPU. Para comprobar solo
esquema y ejecución, define `os.environ["RETAIL_FORECAST_MODE"] = "smoke"` antes
de la preparación. Los resultados sintéticos no miden precisión.

Mantén los siete CSV en `data/raw` del espacio de trabajo o usa el ZIP de Kaggle.
No escribas credenciales en celdas ni salidas. Conserva el notebook ejecutado y
el directorio de resultados antes de desconectar Colab.

In [ ]:
# Pinned source setup; Colab preserves its compatible resident libraries.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "af4d384e8fa47b6034f76aec16e3b598c4fe0d98"
MODELS = False
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT",
    Path("/content") if IN_COLAB else (
        Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
    ),
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import resource
import time

import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import (
    capture_inputs,
    finish_run,
    read_completed_run,
    run_result,
    write_json_artifact,
)
from retail_forecast.config import PROJECT_HORIZON, load_config
from retail_forecast.data import (
    EXPECTED_KAGGLE_SHA256,
    load_frames,
    required_files_exist,
    validate_dataset,
    write_synthetic_dataset,
)
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.features import (
    FeatureSpec,
    build_fold_feature_batches,
    feature_manifest,
    fit_preprocessor,
    model_input_contract,
    transform_features,
)
from retail_forecast.notebook_runs import notebook_config_path, prepare_notebook_run
from retail_forecast.splits import (
    make_rolling_origin_folds,
    split_manifest,
)
from retail_forecast.versions import capture_versions

RUN_STARTED = time.perf_counter()
MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
# The information regime is explicit; results from different policies never pool.
POLICY = os.getenv("RETAIL_FORECAST_POLICY", "asof_v1").strip().lower()
config = load_config(notebook_config_path(MODE, POLICY))
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
run_context, config = prepare_notebook_run(config, kind="features")
print("Run ID:", run_context.run_id, "| root:", run_context.root)
spec = FeatureSpec(config.data.availability_policy, horizon=config.forecast.horizon)
print(
    f"mode={MODE} | policy={spec.availability_policy} "
    f"| horizon={config.forecast.horizon} | feature_spec={spec.sha256[:12]}"
)

## Carga y validación

Se reutilizan los siete CSV o el ZIP del espacio de trabajo. El modo completo valida los hashes esperados antes de construir variables.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir,
        (WORKSPACE_ROOT / "data", WORKSPACE_ROOT),
        token_reader=getpass,
    )
capture_inputs(run_context, input_dir=config.paths.raw_data_dir)
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
dataset_summary = validate_dataset(config)
frames = load_frames(config.paths.raw_data_dir)
display(dataset_summary.to_dict())

## Unidad de cada fila

Una serie se identifica por `(store_nbr, family)`. Una fila del modelo tabular añade origen, fecha objetivo y horizonte. Los 16 horizontes comparten la misma historia observada al origen: no se desplazan los rezagos sobre ventas reales del periodo de validación.

In [ ]:
folds = make_rolling_origin_folds(config, frames["train.csv"]["date"])
split_data = split_manifest(config, folds)
METRICS_DIR = run_context.path("metrics")
write_json_artifact(run_context, "metrics/split_manifest.json", split_data, kind="split_manifest")
split_path = METRICS_DIR / "split_manifest.json"
display(pd.DataFrame([fold.to_dict() for fold in folds]))
print(f"split manifest: {split_path} | sha256={split_data['sha256']}")

## Información admitida

Los rezagos de ventas usan 1, 7, 14, 28 y 56 días respecto al origen; los resúmenes usan ventanas de 7, 14, 28 y 56 días. Ventas y transacciones terminan en el origen.

En `asof_v1` se excluye el petróleo y toda fila de calendario de tipo `Event`. Se admiten calendario determinista, otros festivos tratados como programados y promociones históricas como aproximación al plan comercial. Estas tablas no tienen versiones históricas del plan ni del calendario. La política `competition_v1`, con petróleo de fechas futuras y eventos, es un análisis distinto; sus resultados no se mezclan.

In [ ]:
fold_runs = []
example_batch = None
example_fitted = None
example_transformed = None
for fold in folds:
    started = time.perf_counter()
    training_batch, validation_batch = build_fold_feature_batches(
        frames,
        fold,
        spec,
        snapshot_id=run_context.manifest["snapshot_id"],
        data_source=config.data.source,
    )
    fitted = fit_preprocessor(training_batch, fold.origin)
    transformed_train = transform_features(training_batch, fitted)
    transformed_validation = transform_features(validation_batch, fitted)
    elapsed = time.perf_counter() - started
    raw_bytes = int(
        training_batch.frame.memory_usage(deep=True).sum()
        + validation_batch.frame.memory_usage(deep=True).sum()
    )
    transformed_bytes = int(
        transformed_train.memory_usage(deep=True).sum()
        + transformed_validation.memory_usage(deep=True).sum()
    )
    fold_runs.append(
        {
            "fold": fold.name,
            "origin": fold.origin.date().isoformat(),
            "training_feature_origin": training_batch.origin.date().isoformat(),
            "preprocessor_fitted_through": fitted.fitted_through.date().isoformat(),
            "training_rows": len(training_batch.frame),
            "validation_rows": len(validation_batch.frame),
            "feature_count": validation_batch.feature_count,
            "raw_feature_bytes": raw_bytes,
            "transformed_bytes": transformed_bytes,
            "elapsed_seconds": round(elapsed, 4),
            "process_max_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            "preprocessor_sha256": fitted.sha256,
            "validation_dates": validation_batch.frame["target_date"].nunique(),
            "validation_series": validation_batch.frame[["store_nbr", "family"]]
            .drop_duplicates()
            .shape[0],
            "duplicate_row_keys": int(
                validation_batch.frame.duplicated(
                    ["store_nbr", "family", "forecast_origin", "target_date", "horizon"]
                ).sum()
            ),
            "missing_before_transform": int(validation_batch.frame.isna().sum().sum()),
            "missing_after_transform": int(transformed_validation.isna().sum().sum()),
            "history_missing_rates": {
                column: round(float(validation_batch.frame[column].mean()), 6)
                for column in validation_batch.numeric_features
                if column.startswith(("sales_", "transactions_")) and column.endswith("_missing")
            },
            "known_future_missing_rates": {
                "onpromotion": round(float(validation_batch.frame["onpromotion"].isna().mean()), 6),
                "oil_target": (round(float(validation_batch.frame["oil_target"].isna().mean()), 6)
                    if "oil_target" in validation_batch.frame else None),
            },
            "unknown_category_rates": {
                column: round(float(transformed_validation[f"cat__{column}"].eq(0).mean()), 6)
                for column in validation_batch.categorical_features
            },
            "vocabulary_sizes": {
                key: len(values) for key, values in fitted.categorical_vocabulary.items()
            },
        }
    )
    example_batch = validation_batch
    example_fitted = fitted
    example_transformed = transformed_validation
display(pd.DataFrame(fold_runs))

## Transformaciones del modelo tabular

Imputación y vocabularios categóricos se ajustan únicamente con entrenamiento. La GRU tendrá su propio preprocesamiento en el notebook 04; este notebook no aplica su estandarización por serie al árbol.

In [ ]:
assert example_batch is not None
assert example_fitted is not None
assert example_transformed is not None
preprocessing_evidence = pd.DataFrame(
    {
        "value": [
            example_batch.feature_count,
            len(example_batch.numeric_features),
            len(example_batch.categorical_features),
            int(example_batch.frame.isna().sum().sum()),
            int(example_transformed.isna().sum().sum()),
            example_fitted.fitted_through.date().isoformat(),
        ]
    },
    index=[
        "features",
        "numeric features",
        "categorical features",
        "missing before fold-fit transform",
        "missing after fold-fit transform",
        "preprocessor fitted through",
    ],
)
display(preprocessing_evidence)

## Pruebas de filtraciones

Las comprobaciones deben detectar objetivos futuros usados como historia, cruces entre entrenamiento y validación, cambios del horizonte de 16 días y transformaciones ajustadas después del corte.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_splits.py", "tests/test_features.py"],
    check=False,
    capture_output=True,
    text=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr)
if test_result.returncode != 0:
    raise RuntimeError("Leakage tests failed; model development is blocked.")
print("Leakage and split tests: PASS")

## Entrada de los modelos posteriores

El contrato tabular contiene una fila por serie, origen y horizonte, valores numéricos codificados y categorías aprendidas en entrenamiento. Se conservan identidades de filas, cortes y transformaciones para comparar las mismas observaciones.

In [ ]:
assert example_batch is not None
assert example_transformed is not None
gate_checks = {
    "real horizon is 16": PROJECT_HORIZON == 16 == config.forecast.horizon,
    "two or three folds": 2 <= len(folds) <= 3,
    "every validation window has 16 dates": all(len(fold.validation_dates) == 16 for fold in folds),
    "every validation grid is complete": all(
        run["validation_rows"] == dataset_summary.series * PROJECT_HORIZON
        and run["validation_series"] == dataset_summary.series
        and run["duplicate_row_keys"] == 0
        for run in fold_runs
    ),
    "preprocessors stop at each origin": all(
        run["preprocessor_fitted_through"] == run["origin"] for run in fold_runs
    ),
    "no transformed missing values": all(run["missing_after_transform"] == 0 for run in fold_runs),
    "split manifest exists": split_path.is_file(),
    "processed panel not exported": True,
    "leakage tests passed": test_result.returncode == 0,
}
contract = model_input_contract(example_batch)
test_output_lines = [line for line in test_result.stdout.splitlines() if line.strip()]
run_evidence = {
    "mode": MODE,
    "source": dataset_summary.source,
    "dataset_summary": dataset_summary.to_dict(),
    "official_kaggle_hashes_verified": MODE == "full",
    "official_input_sha256": EXPECTED_KAGGLE_SHA256 if MODE == "full" else None,
    "runtime_versions": capture_versions(),
    "total_elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 4),
    "process_max_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    "leakage_test_files": ["tests/test_splits.py", "tests/test_features.py"],
    "leakage_test_summary": test_output_lines[-1] if test_output_lines else None,
    "gate_checks": gate_checks,
    "gate_status": "PASS" if all(gate_checks.values()) else "FAIL",
}
manifest = feature_manifest(spec, split_data["sha256"], fold_runs, example_batch, run_evidence)
write_json_artifact(run_context, "metrics/feature_manifest.json", manifest, kind="feature_manifest")
feature_path = METRICS_DIR / "feature_manifest.json"
display(
    {
        "row_key": contract["row_key"],
        "tabular_feature_count": manifest["feature_count"],
        "recurrent_target_shape": contract["recurrent"]["target_shape"],
        "feature_manifest_sha256": manifest["sha256"],
        "processed_dataset_exported": manifest["processed_dataset_exported"],
    }
)
print(f"feature manifest: {feature_path}")

In [ ]:
final_gate_checks = {**gate_checks, "feature manifest exists": feature_path.is_file()}
gate_table = pd.DataFrame(
    [
        {"check": name, "status": "PASS" if passed else "FAIL"}
        for name, passed in final_gate_checks.items()
    ]
)
display(gate_table)
if not all(final_gate_checks.values()):
    raise RuntimeError("G3 FAILED: do not continue to model development.")
if MODE == "full":
    print("G3 FULL PASS: feature and rolling-origin contracts are ready for later model work.")
else:
    print("G3 SMOKE PASS: contracts work; reproduce full mode before model development.")
finish_run(run_context)
verified_run = read_completed_run(
    run_context.root,
    expected_mode=MODE,
    expected_source=config.data.source,
)
print("Verified completed run:", run_result(run_context))

## Resultado de esta preparación

Las tablas y pruebas anteriores describen los cortes que usarán los modelos. Una ejecución sintética comprueba funcionamiento; para evaluar precisión hay que ejecutar en `full` con los datos de la competencia.